# 01 - Parametric simulation: ACCIM custom model

This notebook develops a custom-model ACCIM parametric simulation from output discovery to batch execution and quick result inspection.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `ParametricSimulation`
- `ParametricSimulation.discover_available_outputs`
- `ParametricSimulation.apply_outputs_preflight`
- `ParametricSimulation.set_parameters`
- `ParametricSimulation.run_parametric_simulation`
- `WorkspaceArtifactCleaner`
- `ef.get_building`
- `parametric.discover_available_outputs`
- `parametric.select_outputs`
- `parametric.apply_outputs_preflight`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## ACCIM custom parametric: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

`PROCESSES` is the number of simulations run in parallel: half of the logical CPUs by default.

In [1]:
import os
from pathlib import Path

import accim
import pandas as pd
from besos import eppy_funcs as ef
from accim.utils import WorkspaceArtifactCleaner
from accim.parametric_and_optimisation.main import ParametricSimulation
from accim.parametric_and_optimisation.objectives import average_results

pd.set_option("display.max_columns", 160)

# Simulations run in parallel: half of the logical CPUs.
PROCESSES = max(1, (os.cpu_count() or 2) // 2)


## ACCIM custom parametric: set project paths and discover IDF/EPW files

Here we declare the baseline project assets (IDFs, EPWs, and output folder).

For ACCIM custom parametric, this is the configuration anchor. If asset names change, update this block before touching execution cells.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_list`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `idf_list`, `epw_list`, `RESULTS_DIRNAME`.

In [2]:
PROJECT_ROOT = Path.cwd()

idf_list = [
    "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf",
    "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_low_performance.idf",
]
epw_list = ["Seville.epw", "Sydney.epw"]
RESULTS_DIRNAME = "nb01_parametric_custom_results"

print("IDF files found:", [(PROJECT_ROOT / idf).exists() for idf in idf_list])
print("EPW files found:", [(PROJECT_ROOT / epw).exists() for epw in epw_list])


IDF files found: [True, True]
EPW files found: [True, True]


## ACCIM custom parametric: capture baseline workspace state for cleanup

We initialize artifact tracking so generated files can be reviewed and cleaned in a controlled way.

This is especially useful in iterative notebook sessions where repeated runs can quickly accumulate temporary folders and CSV files.

Key arguments used in this cell and why they matter:
- `workspace_root`: defines this setting for the current call.

Variables created in this cell:
- `artifact_cleaner`: reused in subsequent workflow blocks.
- `baseline_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `artifact_cleaner`, `baseline_files`.

In [3]:
artifact_cleaner = WorkspaceArtifactCleaner(workspace_root=PROJECT_ROOT)
baseline_files = artifact_cleaner.capture_initial_state()
print("Baseline file count:", len(baseline_files))


Baseline file count: 14


## ACCIM custom parametric: load the baseline IDF model

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution.

Variables created in this cell:
- `building`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `building`.

In [4]:
building = ef.get_building(idf_list[0])


## ACCIM custom parametric: apply runtime-reduction settings

Apply runtime reductions to keep notebook runs faster during exploration.

This block shortens exploratory runs by reducing simulation burden. Use it for prototyping and switch to fuller settings when producing final results.

`reduce_runtime` also changes the run period, to 1 January by default, so its run period arguments keep the full year: the results stay annual. `Run_Simulation_for_Sizing_Periods = "No"` keeps the design days of the sample model out of the annual results (they are still used to size the HVAC system).

Key arguments used in this cell and why they matter:
- `idf_object`: defines this setting for the current call.
- `timesteps`: defines this setting for the current call.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [5]:
accim.utils.reduce_runtime(
    idf_object=building,
    timesteps=2,
    runperiod_begin_month=1,
    runperiod_begin_day_of_month=1,
    runperiod_end_month=12,
    runperiod_end_day_of_month=31,
)
building.idfobjects["SimulationControl"][0].Run_Simulation_for_Sizing_Periods = "No"


Solar distribution has been set to MinimalShadowing.
Shading Calculation Update Frequency was previously set to 20 days, and it has been modified to 20 days.
Maximum Figures in Shadow Overlap Calculations was previously set to 15000 days, and it has been modified to 200 days.
Number of Timesteps per Hour was previously set to 6 days, and it has been modified to 2 days.


## ACCIM custom parametric: initialize parametric simulation object

Create the parametric simulation object with weather context and model settings.

This creates the parametric simulation object that will orchestrate outputs, parameters, and run plans. From here on, the workflow uses this object as the main controller.

Key arguments used in this cell and why they matter:
- `building`: loaded IDF model used in this workflow.
- `parameters_type`: parameter family used in the simulation setup.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs.

Variables created in this cell:
- `parametric`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `parametric`.

In [6]:
parametric = ParametricSimulation(
	building=building,
	parameters_type="accim custom model",
	epws=epw_list,
	output_freqs=["hourly"],
)



--------------------------------------------------------
Adaptive-Comfort-Control-Implemented Model (ACCIM) v0.7.8
--------------------------------------------------------

This tool allows to apply adaptive setpoint temperatures. 
For further information, please read the documentation: 
https://accim.readthedocs.io/en/master/
For a visual understanding of the tool, please visit the following jupyter notebooks:
-    Using addAccis() to apply adaptive setpoint temperatures
https://accim.readthedocs.io/en/master/jupyter_notebooks/addAccis/using_addAccis.html
-    Using rename_epw_files() to rename the EPWs for proper data analysis after simulation
https://accim.readthedocs.io/en/master/jupyter_notebooks/rename_epw_files/using_rename_epw_files.html
-    Using runEp() to directly run simulations with EnergyPlus
https://accim.readthedocs.io/en/master/jupyter_notebooks/runEp/using_runEp.html
-    Using the class Table() for data analysis
https://accim.readthedocs.io/en/master/jupyter_notebo

Generic contaminant setpoint: Always 0.5ppm Schedule has been added


Air distribution effectiveness (always 1) Schedule has been added
VRF Heating Cooling (Northern Hemisphere) Schedule has been added
DefaultFanEffRatioCurve Curve:Cubic Object has been added
VRFTUCoolCapFT Curve:Cubic Object has been added
VRFTUHeatCapFT Curve:Cubic Object has been added
VRFCoolCapFTBoundary Curve:Cubic Object has been added
VRFCoolEIRFTBoundary Curve:Cubic Object has been added
CoolingEIRLowPLR Curve:Cubic Object has been added
VRFHeatCapFTBoundary Curve:Cubic Object has been added
VRFHeatEIRFTBoundary Curve:Cubic Object has been added
HeatingEIRLowPLR Curve:Cubic Object has been added
DefaultFanPowerRatioCurve Curve:Exponent Object has been added
DXHtgCoilDefrostEIRFT Curve:Biquadratic Object has been added
VRFCoolCapFT Curve:Biquadratic Object has been added
VRFCoolCapFTHi Curve:Biquadratic Object has been added
VRFCoolEIRFT Curve:Biquadratic Object has been added
VRFCoolEIRFTHi Curve:Biquadratic Object has been added
VRFHeatCapFT Curve:Biquadratic Object has been ad

Added - 46 per-ComfStand SetAST subprograms
Added - SetAST Program (base program that calls SetAST_Master)
Added - SetASTnoTol Program
Added - CountHoursNoApp_Floor_1_Residential_Living_Occupants Program
Added - SetGeoVarFloor_1_Residential_Living_Occupants Program
Added - CountHoursNoApp_Floor_2_Residential_Living_Occupants Program
Added - SetGeoVarFloor_2_Residential_Living_Occupants Program
Added - SetInputData Program
Added - SetVOFinputData Program
Added - SetVST Program
Added - ApplyAST_Floor_1_Residential_Living_Occupants Program
Added - ApplyAST_Floor_2_Residential_Living_Occupants Program
Added - Comfort Temperature Output Variable
Added - Adaptive Cooling Setpoint Temperature Output Variable
Added - Adaptive Heating Setpoint Temperature Output Variable
Added - Adaptive Cooling Setpoint Temperature_No Tolerance Output Variable
Added - Adaptive Heating Setpoint Temperature_No Tolerance Output Variable
Added - z_test_ComfStand Output Variable
Added - z_test_ACSTaul Output Variab

## ACCIM custom parametric: discover available meter and variable outputs

Inspect available meters and variables before selecting valid outputs.

Before selecting outputs, we inspect what meters and variables are currently available. That discovery step reduces mistakes when requesting objective or analysis outputs.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.

Variables created in this cell:
- `df_meters_available`: reused in subsequent workflow blocks.
- `df_vars_available`: reused in subsequent workflow blocks.
- `outputs_meta`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that the discovered meter/variable tables are not unexpectedly empty.
- Confirm the reported discovery source matches your preferred backend.

In [7]:
available_outputs = parametric.discover_available_outputs(
	reduce_sim_time=True,
	prefer="testsimeplus",
)
df_meters_available = available_outputs["meters"]
df_vars_available = available_outputs["variables"]
outputs_meta = available_outputs["meta"]

print("Discovery source:", outputs_meta.get("source"))
print("Meters available:", len(df_meters_available))
print("Variables available:", len(df_vars_available))


Solar distribution is already set to MinimalShadowing, therefore no action has been performed.
Shading Calculation Update Frequency was previously set to 20 days, and it has been modified to 20 days.
Maximum Figures in Shadow Overlap Calculations was previously set to 200 days, and it has been modified to 200 days.
Number of Timesteps per Hour was previously set to 2 days, and it has been modified to 2 days.


Discovery source: testsimeplus
Meters available: 6
Variables available: 144


## ACCIM custom parametric: select objective and analysis outputs

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `meters`: energy meters to include as outputs.
- `match`: matching rule used to select outputs.
- `on_missing`: behavior when a requested output is missing.
- `suggest`: shows suggestions when no exact match is found.
- `variables`: output variables to include by name or pattern.

Variables created in this cell:
- `objective_meters`: reused in subsequent workflow blocks.
- `auxiliary_variables`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.
- `_`: reused in subsequent workflow blocks.
- `df_vars_sel`: reused in subsequent workflow blocks.
- `vars_report`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [8]:
objective_meters = ["Heating:Electricity", "Cooling:Electricity"]
auxiliary_variables = [
	"Zone Operative Temperature",
	"Running Average Outdoor Air Temperature",
]

selection = parametric.select_outputs(
	meters=objective_meters,
	match="case_insensitive",
	on_missing="raise",
	suggest=True,
)
df_meters_sel = selection["meters"]

selection = parametric.select_outputs(
	variables=auxiliary_variables,
	match="contains",
	on_missing="warn",
	suggest=True,
)
df_vars_sel = selection["variables"]
vars_report = selection["report"]

if vars_report["missing"]["variables"]:
	print("Missing auxiliary variables:", vars_report["missing"]["variables"])


## ACCIM custom parametric: run output preflight checks before execution

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

`set_output_readers` turns each selected meter and variable into one output of the study. Without a `name` column, each output is named after its variable, so the variables of the two zones would share a name; without a `func` column, the hourly values are summed, which suits energy but not temperatures. The variable readers are therefore named after the variable and its key (the zone or occupant group), and `average_results` turns the hourly values into an annual mean.

Key arguments used in this cell and why they matter:
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `on_missing`: behavior when a requested output is missing.
- `df_output_meter`: final meter table used by the simulation.
- `df_output_variable`: final variable table used by the simulation.
- `name`, `func`: name of each output and function applied to its hourly values.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [9]:
parametric.apply_outputs_preflight(
	df_meters_sel=df_meters_sel,
	df_vars_sel=df_vars_sel,
	clean_mode="all",
	validate_before_apply=False,
	validate_after_apply=True,
	on_missing="raise",
)
# One reader per zone (or occupant group), named after the variable and its key, which
# averages the hourly values; the meters keep the default sum of the hourly values.
readers_vars = df_vars_sel.copy()
readers_vars["name"] = readers_vars["variable_name"] + " (" + readers_vars["key_value"] + ")"
readers_vars["func"] = [average_results] * len(readers_vars)

parametric.set_output_readers(df_output_meter=df_meters_sel, df_output_variable=readers_vars)
print("Outputs:", df_meters_sel["key_name"].tolist() + readers_vars["name"].tolist())


Outputs: ['Heating:Electricity', 'Cooling:Electricity', 'Zone Operative Temperature (FLOOR_1_ZONE)', 'Zone Operative Temperature (FLOOR_2_ZONE)', 'Zone Thermal Comfort ASHRAE 55 Adaptive Model Running Average Outdoor Air Temperature (FLOOR_1 RESIDENTIAL LIVING OCCUPANTS)', 'Zone Thermal Comfort CEN 15251 Adaptive Model Running Average Outdoor Air Temperature (FLOOR_1 RESIDENTIAL LIVING OCCUPANTS)', 'Zone Thermal Comfort ASHRAE 55 Adaptive Model Running Average Outdoor Air Temperature (FLOOR_2 RESIDENTIAL LIVING OCCUPANTS)', 'Zone Thermal Comfort CEN 15251 Adaptive Model Running Average Outdoor Air Temperature (FLOOR_2 RESIDENTIAL LIVING OCCUPANTS)']


## ACCIM custom parametric: define decision variables and objective directions

Set parameter ranges or values and build the study problem.

Here we define the decision space and objective orientation used in simulation or optimization. These choices directly determine how the final result space will look.

Key arguments used in this cell and why they matter:
- `accis_params_dict`: ACCIS/aPMV parameter definition used in the study.

Variables created in this cell:
- `accis_parameters`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `accis_parameters`.

In [10]:
accis_parameters = {
	"CustAST_m": (0.15, 0.95),
	"CustAST_n": (8, 23),
	"CustAST_ASToffset": (1.0, 6.0),
	"CustAST_ASTall": (8, 16),
	"CustAST_ASTaul": (28, 38),
}

parametric.set_parameters(accis_params_dict=accis_parameters)
parametric.set_problem()


## ACCIM custom parametric: draw a Latin hypercube sample

`sampling_lhs` draws `num_samples` combinations of the parameter ranges with a Latin hypercube, and pairs each one with every EPW, so the plan has `num_samples x len(epw_list)` rows. The Latin hypercube of accim has no seed argument: a new sample is drawn every time the cell runs.

The ranges of m and n are wide, so some samples give comfort temperatures well above or below those of published adaptive models, and the results show it (annual mean operative temperatures above 30 °C, for instance). `set_mn_filter` keeps the samples inside the band of published models: notebook wf1 shows it.

Key arguments used in this cell and why they matter:
- `num_samples`: number of samples.

Variables created in this cell:
- `lhs_df`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm that the plan has `num_samples x len(epw_list)` rows.

In [11]:
parametric.sampling_lhs(num_samples=4)
lhs_df = parametric.parameters_values_df.copy()
print("LHS rows:", len(lhs_df))
lhs_df.head()


LHS rows: 8


,CustAST_m,CustAST_n,CustAST_ASToffset,CustAST_ASTall,CustAST_ASTaul,epw
0,0.517344,20.569361,5.274068,14.301510,28.130404,Seville.epw
1,0.585588,9.448885,2.820583,11.292354,34.019490,Seville.epw
2,0.827407,18.241273,4.394507,8.492577,37.558045,Seville.epw
3,0.223251,12.667872,2.183316,12.346452,31.781236,Seville.epw
4,0.517344,20.569361,5.274068,14.301510,28.130404,Sydney.epw


## ACCIM custom parametric: build full-factorial plan

Prepare the sampling strategy or explicit dataframe of cases to run.

This cell builds the dataframe of cases to execute, either sampled or manually assembled. The generated table is the direct input for the execution stage.

Key arguments used in this cell and why they matter:
- `level`: number of levels per parameter in full factorial.

Variables created in this cell:
- `ff_df`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `ff_df`.

In [12]:
parametric.sampling_full_factorial(level=2)
ff_df = parametric.parameters_values_df.copy()
print("Full factorial rows:", len(ff_df))
print(ff_df.head())


Full factorial rows: 64
   CustAST_m  CustAST_n  CustAST_ASToffset  CustAST_ASTall  CustAST_ASTaul  \
0       0.15        8.0                1.0             8.0            28.0   
1       0.55        8.0                1.0             8.0            28.0   
2       0.15       15.5                1.0             8.0            28.0   
3       0.55       15.5                1.0             8.0            28.0   
4       0.15        8.0                3.5             8.0            28.0   

           epw  
0  Seville.epw  
1  Seville.epw  
2  Seville.epw  
3  Seville.epw  
4  Seville.epw  


## ACCIM custom parametric: assemble the final execution dataframe

Prepare the sampling strategy or explicit dataframe of cases to run.

This cell builds the dataframe of cases to execute, either sampled or manually assembled. The generated table is the direct input for the execution stage.

Variables created in this cell:
- `plan_df`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `plan_df`.

In [13]:
# custom_plan = pd.DataFrame(
#     [
#         {"CustAST_m": 0.30, "CustAST_n": 12, "CustAST_ASToffset": 2.5, "CustAST_ASTall": 10, "CustAST_ASTaul": 32, "epw": epw_list[0]},
#         {"CustAST_m": 0.60, "CustAST_n": 18, "CustAST_ASToffset": 4.0, "CustAST_ASTall": 12, "CustAST_ASTaul": 35, "epw": epw_list[1]},
#     ]
# )
# parametric.sampling_custom(custom_plan)
# plan_df = parametric.parameters_values_df.copy()

# Use the LHS sample by default (only the first IDF is loaded; notebook 06 shows multi-IDF runs).
plan_df = lhs_df.copy()


## ACCIM custom parametric: estimate case count from plan and EPW multipliers

Before execution, we estimate workload size from plan cardinality and solver budget settings.

Use this checkpoint to align runtime expectations with available compute resources before launching.

Only the first IDF of `idf_list` is loaded in this notebook, so each row of the plan is one simulation per EPW; notebook 06 runs several IDFs.

Variables created in this cell:
- `epw_multiplier`: reused in subsequent workflow blocks.
- `idf_multiplier`: reused in subsequent workflow blocks.
- `estimated_simulations`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare estimated run count with your available compute/time budget.
- Adjust evaluations, population size, or climate count if the estimate is too high.

In [14]:
epw_multiplier = 1 if "epw" in plan_df.columns else len(epw_list)
idf_multiplier = 1 if "idf" in plan_df.columns else len(parametric.buildings)
estimated_simulations = len(plan_df) * epw_multiplier * idf_multiplier

print("Plan rows:", len(plan_df))
print("EPW multiplier:", epw_multiplier)
print("IDF multiplier:", idf_multiplier)
print("Estimated simulation cases:", estimated_simulations)


Plan rows: 8
EPW multiplier: 1
IDF multiplier: 1
Estimated simulation cases: 8


## ACCIM custom parametric: execute parametric run and inspect output rows

This is the execution checkpoint for the configured parametric plan.

After the run, verify row counts and core objective columns immediately so issues are caught before post-processing.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `df`: case plan dataframe executed in this run.
- `processes`: number of parallel processes for simulation runs.
- `keep_input`: whether generated input files are preserved.
- `keep_dirs`: whether per-run directories are preserved.

Variables created in this cell:
- `outputs`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare returned row count with the expected number of plan combinations.
- Inspect the first rows to ensure objective columns are present and populated.

In [15]:
outputs = parametric.run_parametric_simulation(
	out_dir=RESULTS_DIRNAME,
	df=plan_df,
	processes=PROCESSES,
	keep_input=True,
	keep_dirs=False,
)
print("Rows:", len(outputs))
print(outputs.head())


  [info] simulation_summary updated for parametric after run_parametric_simulation.
Rows: 8
   Heating:Electricity  Cooling:Electricity  \
0         3.368456e+10         1.487264e+10   
1         6.475242e+09         7.044204e+10   
2         5.202694e+10         4.818671e+09   
3         4.509947e+09         1.125433e+11   
4         2.122351e+10         7.014654e+09   

   Zone Operative Temperature (FLOOR_1_ZONE)  \
0                                  33.795496   
1                                  23.831245   
2                                  35.884467   
3                                  19.126237   
4                                  32.281587   

   Zone Operative Temperature (FLOOR_2_ZONE)  \
0                                  30.386439   
1                                  22.687379   
2                                  32.988950   
3                                  18.617909   
4                                  28.817570   

   Zone Thermal Comfort ASHRAE 55 Adaptive Mode

### Reading the result

Review the row count and confirm it matches the expected combinations from your plan and EPW set.

## ACCIM custom parametric: inspect generated artifacts before cleanup

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `max_items`: maximum number of items shown in logs.

Variables created in this cell:
- `generated_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `generated_files`.

In [16]:
generated_files = artifact_cleaner.print_generated_files(max_items=200)


Generated files detected: 6
 - nb01_parametric_custom_results/accim_idf_backup_OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance_pre_parametric_20261015_214526.idf
 - nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.csv
 - nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.json
 - nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.pkl
 - nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.xlsx
 - nb01_parametric_custom_results/outputs_param_simulation_batches/outputs_param_simulation_batch_00001_20261015_214605_189330.pkl


## ACCIM custom parametric: preview cleanup actions with dry-run

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [17]:
# Force-remove every generated artifact inside the dedicated result folder,
# including generated IDF files.
artifact_cleaner.delete_generated_files(
	allow_patterns=[f"{RESULTS_DIRNAME}/**"],
	deny_patterns=[],
	dry_run=True,
	remove_empty_dirs=True,
)


Files planned for deletion: 6
 - nb01_parametric_custom_results/accim_idf_backup_OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance_pre_parametric_20261015_214526.idf
 - nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.csv
 - nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.json
 - nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.pkl
 - nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.xlsx
 - nb01_parametric_custom_results/outputs_param_simulation_batches/outputs_param_simulation_batch_00001_20261015_214605_189330.pkl
Dry run enabled. No files were deleted.


['nb01_parametric_custom_results/accim_idf_backup_OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance_pre_parametric_20261015_214526.idf',
 'nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.csv',
 'nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.json',
 'nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.pkl',
 'nb01_parametric_custom_results/outputs_param_simulation_20261015_214605.xlsx',
 'nb01_parametric_custom_results/outputs_param_simulation_batches/outputs_param_simulation_batch_00001_20261015_214605_189330.pkl']

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

`DELETE_RESULTS` is False by default, so nothing is deleted. Set it to True to delete the files listed above.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [18]:
DELETE_RESULTS = False

if DELETE_RESULTS:
    artifact_cleaner.delete_generated_files(
        allow_patterns=[f"{RESULTS_DIRNAME}/**"],
        deny_patterns=[],
        dry_run=False,
        remove_empty_dirs=True,
    )
